In [1]:
# Install required packages
!pip install -U datasets huggingface_hub fsspec evaluate
!pip install trl accelerate bitsandbytes transformers -q

import random
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
import os
import shutil
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import AutoModelForCausalLM, pipeline
from trl import PPOTrainer, PPOConfig
from accelerate import PartialState
import evaluate

# Fix random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.32.4
    Uninstalling huggingface-hub-0.32.4:
      Successfully uninstalled huggingface-hub-0.32.4
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency 

In [2]:
# Загружаем датасет imdb
dataset_name = "stanfordnlp/imdb"
original_dataset = load_dataset(dataset_name)

# Посмотрим на структуру загруженного датасета
print("Структура исходного датасета:")
print(original_dataset)

# Посмотрим на пример из обучающей выборки
print("\nПример отзыва из обучающей выборки:")
print(original_dataset['train'][0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Структура исходного датасета:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Пример отзыва из обучающей выборки:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and 

In [3]:
import os
# os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
full_train_dataset = original_dataset['train']

# Разделяем на train и validation
# stratify_by_column='label' важен для сохранения баланса классов
train_val_split = full_train_dataset.train_test_split(test_size=0.2, seed=SEED, stratify_by_column='label')

# train_val_split['train'] - наша новая обучающая выборка (80% от исходной train)
# train_val_split['test']  - наша новая валидационная выборка (20% от исходной train)
# original_dataset['test'] - оригинальная тестовая выборка, которую мы пока не используем для обучения reward-модели

# TODO FIX 500
processed_dataset = DatasetDict({
    'train': train_val_split['train'],
    'validation': train_val_split['test'],
    'test': original_dataset['test'] # Оставляем оригинальный тест на всякий случай, если он понадобится
})

print("\nСтруктура обработанного датасета (для reward-модели):")
print(processed_dataset)

print(f"\nРазмер новой обучающей выборки: {len(processed_dataset['train'])}")
print(f"Размер новой валидационной выборки: {len(processed_dataset['validation'])}")
print(f"Размер оригинальной тестовой выборки: {len(processed_dataset['test'])}")

# Проверим примеры
print("\nПример из новой обучающей выборки:")
print(processed_dataset['train'][0])
print("\nПример из новой валидационной выборки:")
print(processed_dataset['validation'][0])


Структура обработанного датасета (для reward-модели):
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})

Размер новой обучающей выборки: 20000
Размер новой валидационной выборки: 5000
Размер оригинальной тестовой выборки: 25000

Пример из новой обучающей выборки:
{'text': 'After reading tons of good reviews about this movie I decided to take it for a spin (I bought it on DVD, hence the "spin" pun...I\'m a dork). The beginning was everything I hoped for, a perfect set-up (along with some quotes that I\'ve heard on Various Wu-Tang albums) to what should have been a good movie. But the plot I heard was so great, was so predictable. Every time I saw a character (except for the Lizard) I guessed which Venom he was. Plus, the only cool character gets killed 

In [5]:
from transformers import AutoTokenizer

# Загружаем токенизатор для distilroberta-base
model_checkpoint = "distilroberta-base"
tokenizer_reward = AutoTokenizer.from_pretrained(model_checkpoint) # Дадим другое имя, чтобы не пересекалось с будущим токенизатором для GPT-2

# Функция для токенизации данных
def tokenize_function_reward(examples):
    # Токенизируем текст, обрезаем до максимальной длины, если нужно
    # max_length для RoBERTa обычно 512
    return tokenizer_reward(examples["text"], truncation=True, padding="max_length", max_length=512)

# Применяем токенизацию ко всем частям нашего датасета (train, validation)
# Мы будем использовать processed_dataset, который мы подготовили ранее
tokenized_datasets_reward = processed_dataset.map(tokenize_function_reward, batched=True)

# Удаляем колонку 'text', так как она больше не нужна (модель будет использовать input_ids, attention_mask)
# И переименовываем 'label' в 'labels' (ожидаемое имя для Hugging Face Trainer)
tokenized_datasets_reward = tokenized_datasets_reward.remove_columns(["text"])
tokenized_datasets_reward = tokenized_datasets_reward.rename_column("label", "labels")
tokenized_datasets_reward.set_format("torch") # Устанавливаем формат данных в torch тензоры

print("\nСтруктура токенизированного датасета для reward-модели:")
print(tokenized_datasets_reward)
print("\nПример элемента из токенизированного обучающего набора (reward):")
# Посмотрим на ключи и содержимое одного элемента
sample_item = tokenized_datasets_reward['train'][0]
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in sample_item.items()})
print(f"Input IDs (часть): {sample_item['input_ids'][:20]}...")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]


Структура токенизированного датасета для reward-модели:
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
})

Пример элемента из токенизированного обучающего набора (reward):
{'labels': torch.Size([]), 'input_ids': torch.Size([512]), 'attention_mask': torch.Size([512])}
Input IDs (часть): tensor([   0, 4993, 2600, 7741,    9,  205, 6173,   59,   42, 1569,   38, 1276,
           7,  185,   24,   13,   10, 6287,   36,  100])...


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Загружаем модель distilroberta-base для классификации последовательностей
# num_labels=2, так как у нас две метки тональности (0 - негативный, 1 - позитивный)
reward_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# Определяем метрику для оценки
# Для задач классификации часто используется accuracy
# load_metric загружает метрику по ее названию
metric_accuracy = evaluate.load("accuracy")

def compute_metrics_reward(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric_accuracy.compute(predictions=predictions, references=labels)

# Определяем аргументы для обучения
# Используем простые и шаблонные параметры
reward_model_output_dir = "./reward_model_distilroberta_imdb"

training_args_reward = TrainingArguments(
    output_dir=reward_model_output_dir,
    num_train_epochs=2,  # Для скорости и простоты начнем с 1 эпохи. Для реальных задач может потребоваться больше.
    per_device_train_batch_size=64, 
    per_device_eval_batch_size=64,  
    warmup_steps=50,               # Количество шагов для "прогрева" learning rate
    weight_decay=0.01,              # Коэффициент L2-регуляризации
    logging_dir=f'{reward_model_output_dir}/logs',
    logging_steps=50,              # Как часто логировать метрики обучения
    eval_strategy="epoch",    # Проводить оценку в конце каждой эпохи
    save_strategy="epoch",          # Сохранять модель в конце каждой эпохи
    load_best_model_at_end=True,    # Загрузить лучшую модель (по метрике на валидации) по итогам обучения
    report_to="none",               # Отключаем логирование в сторонние сервисы (wandb, tensorboard) для простоты
    seed=SEED                       # Для воспроизводимости внутри Trainer
)

# Создаем объект Trainer
trainer_reward = Trainer(
    model=reward_model,
    args=training_args_reward,
    train_dataset=tokenized_datasets_reward["train"],
    eval_dataset=tokenized_datasets_reward["validation"],
    compute_metrics=compute_metrics_reward,
    tokenizer=tokenizer_reward 
)

print("Reward-модель (distilroberta-base) и Trainer инициализированы.")

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<ipython-input-6-2177202442>:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_reward = Trainer(


Reward-модель (distilroberta-base) и Trainer инициализированы.


In [ ]:
import torch

print(f"Обучение будет на устройстве: {device}")
reward_model.to(device) # Явно переместим модель на устройство, хотя Trainer это тоже делает

print("\nНачинаем обучение reward-модели...")
train_results = trainer_reward.train()
print("Обучение reward-модели завершено.")

# Выведем информацию о тренировке
print(f"Логи тренировки: {train_results.metrics}")


print("\nОценка лучшей модели на валидационном наборе:")
eval_results = trainer_reward.evaluate()
print(f"Результаты оценки на валидации: {eval_results}")

final_reward_model_path = f"{reward_model_output_dir}/final"
trainer_reward.save_model(final_reward_model_path)
tokenizer_reward.save_pretrained(final_reward_model_path) # Сохраняем токенизатор рядом с моделью
print(f"\nОбученная reward-модель и токенизатор сохранены в: {final_reward_model_path}")



Обучение будет на устройстве: cuda

Начинаем обучение reward-модели...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.245100,0.200697,0.924600
2,0.139300,0.194886,0.931000


Обучение reward-модели завершено.
Логи тренировки: {'train_runtime': 2082.5613, 'train_samples_per_second': 19.207, 'train_steps_per_second': 0.301, 'total_flos': 5298695946240000.0, 'train_loss': 0.21151570504465803, 'epoch': 2.0}

Оценка лучшей модели на валидационном наборе:


Результаты оценки на валидации: {'eval_loss': 0.19488590955734253, 'eval_accuracy': 0.931, 'eval_runtime': 75.9852, 'eval_samples_per_second': 65.802, 'eval_steps_per_second': 1.04, 'epoch': 2.0}

Обученная reward-модель и токенизатор сохранены в: ./reward_model_distilroberta_imdb/final


In [10]:
import torch
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from trl import PPOTrainer, PPOConfig
from datasets import load_dataset
from accelerate import PartialState


output_dir = "./ppo_positive_imdb"
shutil.rmtree(output_dir, ignore_errors=True)

policy_model_name = "lvwerra/gpt2-imdb"
tokenizer = AutoTokenizer.from_pretrained(policy_model_name, padding_side="left")
tokenizer.add_special_tokens({"pad_token": "[PAD]"})

# Загружаем модели
policy = AutoModelForCausalLM.from_pretrained(policy_model_name)
ref_policy = AutoModelForCausalLM.from_pretrained(policy_model_name)
value_model = AutoModelForSequenceClassification.from_pretrained(policy_model_name, num_labels=1)
sentiment_classifier = AutoModelForSequenceClassification.from_pretrained(final_reward_model_path)
sentiment_classifier.to(device)

# 2. Напиши простую функцию-обертку
def get_reward_from_text(texts):
    # Токенизируем сгенерированный текст
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        # Получаем логиты от классификатора
        logits = sentiment_classifier(**inputs).logits
    # Возвращаем только логит позитивного класса (индекс 1).
    # PPOTrainer использует это как награду. Просто и эффективно.
    return logits[:, 1]


# Перемещаем на устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = policy.to(device)
ref_policy = ref_policy.to(device)
value_model = value_model.to(device)
reward_model = reward_model.to(device)

# Блок 2.5: Подготовка датасета для PPO
ppo_dataset = load_dataset("stanfordnlp/imdb", split="train")
ppo_dataset = ppo_dataset.select(range(200))  # Маленький набор для теста

def prepare_dataset(dataset, tokenizer):
    def tokenize(element):
        # Используем начало отзывов как промпты
        prompts = [text[:50] + "..." for text in element['text']]
        outputs = tokenizer(prompts, padding=False)
        return {"input_ids": outputs["input_ids"]}

    return dataset.map(
        tokenize,
        batched=True,
        remove_columns=dataset.column_names,
    )

with PartialState().local_main_process_first():
    train_dataset = prepare_dataset(ppo_dataset.select(range(180)), tokenizer)
    eval_dataset = prepare_dataset(ppo_dataset.select(range(180, 200)), tokenizer)

# Блок 2.6: Конфигурация PPO
ppo_config = PPOConfig(
    output_dir=output_dir,
    learning_rate=1.41e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    total_episodes=100,  # Мало эпизодов для теста
    num_ppo_epochs=4,
    num_mini_batches=1,
    seed=42,
    report_to="none",
    remove_unused_columns=False,
)

# Блок 2.7: Создание trainer
trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=policy,
    ref_model=ref_policy,
    reward_model=get_reward_from_text,
    value_model=value_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Блок 2.8: Генерация до обучения
test_prompts = [
    "This movie was",
    "I watched this film and",
    "The acting in this movie",
    "The plot was",
    "Overall, I think"
]

print("=== BEFORE PPO TRAINING ===")
initial_generations = {}
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = policy.generate(**inputs, max_length=80, num_return_sequences=1,
                                 temperature=0.8, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    initial_generations[prompt] = text
    print(f"\nPrompt: {prompt}")
    print(f"Generation: {text}")

# Блок 2.9: Обучение PPO
print("\n\n=== STARTING PPO TRAINING ===")
trainer.train()
trainer.save_model(output_dir)

# Блок 2.10: Генерация после обучения
print("\n\n=== AFTER PPO TRAINING ===")
final_generations = {}
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = policy.generate(**inputs, max_length=80, num_return_sequences=1,
                                 temperature=0.8, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    final_generations[prompt] = text
    print(f"\nPrompt: {prompt}")
    print(f"Generation: {text}")

# Блок 2.11: Анализ результатов
print("\n\n=== SENTIMENT ANALYSIS ===")
from transformers import pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model=final_reward_model_path,
                            device=0 if torch.cuda.is_available() else -1)

for prompt in test_prompts:
    before = sentiment_analyzer(initial_generations[prompt])[0]
    after = sentiment_analyzer(final_generations[prompt])[0]

    print(f"\nPrompt: '{prompt}'")
    print(f"Before: {before['label']} (score: {before['score']:.3f})")
    print(f"After: {after['label']} (score: {after['score']:.3f})")

    # LABEL_1 = позитивный, LABEL_0 = негативный
    improved = (after['label'] == 'LABEL_1' and
                (before['label'] != 'LABEL_1' or after['score'] > before['score']))
    print(f"Improved: {'✓' if improved else '✗'}")

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at lvwerra/gpt2-imdb and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AttributeError: 'function' object has no attribute 'modules'